# Monitoreo y detección de data drift

Este notebook documenta la estrategia de seguimiento continuo una vez el modelo está en producción. Se enfoca en identificar desviaciones en la distribución de datos (data drift) y en el desempeño predictivo, cumpliendo la rúbrica del proyecto final.


## Objetivos
- Cargar el modelo y pipeline de features desplegados.
- Simular un escenario de referencia vs. producción usando particiones del dataset.
- Calcular métricas de data drift (KS, PSI, Jensen-Shannon, Chi-cuadrado) y generar visualizaciones.
- Documentar recomendaciones de monitoreo continuo y posibles alertas.
- Proveer fragmentos reutilizables para integrarlos en una aplicación Streamlit.


In [ ]:
# Importación de librerías y configuración
import json
import sys
from pathlib import Path
from typing import Dict, List

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

SRC_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = SRC_DIR.parent.parent
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from ft_engineering import (
    clean_column_names,
    infer_variable_roles,
    load_project_config,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)



In [ ]:
# Carga de datos, pipeline y modelo entrenado
config = load_project_config()
data_path = PROJECT_ROOT / config["pipeline_config"]["data_path"]
raw_df = pd.read_csv(data_path, sep=";", encoding="utf-8")
raw_df = clean_column_names(raw_df)

# Preparación de columnas auxiliares
raw_df["Target_id"] = raw_df["Target"].map({"Dropout": 0, "Enrolled": 1, "Graduate": 2})
numeric_cols, categorical_cols = infer_variable_roles(raw_df, target_col="Target")

reference_df, current_df = train_test_split(
    raw_df,
    test_size=config["model_config"]["test_size"],
    random_state=config["model_config"]["random_state"],
    stratify=raw_df["Target"],
)

outputs_dir = PROJECT_ROOT / "mlops_pipeline" / "outputs"
models_dir = PROJECT_ROOT / "mlops_pipeline" / "models"

feature_pipeline = joblib.load(outputs_dir / "feature_pipeline.joblib")
metadata_path = models_dir / "model_metadata.json"
with metadata_path.open("r", encoding="utf-8") as fp:
    metadata = json.load(fp)

model_path = models_dir / f"best_model_{metadata['model_name']}.joblib"
trained_model = joblib.load(model_path)

reference_X = feature_pipeline.transform(reference_df.drop(columns=["Target"]))
current_X = feature_pipeline.transform(current_df.drop(columns=["Target"]))
reference_y = reference_df["Target"]
current_y = current_df["Target"]

print(
    "Datos cargados:",
    f"Referencia: {reference_df.shape} | Producción simulada: {current_df.shape}",
    f"Modelo: {metadata['model_name']}",
    sep="\n",
)


In [ ]:
# Evaluación de desempeño en producción simulada
y_pred_current = trained_model.predict(current_X)
y_pred_reference = trained_model.predict(reference_X)

perf_reference = accuracy_score(reference_y, y_pred_reference)
perf_current = accuracy_score(current_y, y_pred_current)

print(f"Accuracy referencia: {perf_reference:.3f}")
print(f"Accuracy producción simulada: {perf_current:.3f}")

classification_report_current = pd.DataFrame(
    classification_report(current_y, y_pred_current, output_dict=True)
).T
classification_report_current


In [ ]:
# Funciones para cálculo de métricas de data drift
def kolmogorov_smirnov(reference: pd.Series, current: pd.Series) -> float:
    stat, _ = stats.ks_2samp(reference.dropna(), current.dropna())
    return stat


def population_stability_index(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    ref, edges = np.histogram(reference.dropna(), bins=bins)
    cur, _ = np.histogram(current.dropna(), bins=edges)

    ref_perc = np.clip(ref / ref.sum(), 1e-6, None)
    cur_perc = np.clip(cur / cur.sum(), 1e-6, None)

    psi = np.sum((ref_perc - cur_perc) * np.log(ref_perc / cur_perc))
    return float(psi)


def jensen_shannon_distance(reference: pd.Series, current: pd.Series, bins: int = 30) -> float:
    ref_hist, edges = np.histogram(reference.dropna(), bins=bins, density=True)
    cur_hist, _ = np.histogram(current.dropna(), bins=edges, density=True)
    return float(jensenshannon(ref_hist + 1e-8, cur_hist + 1e-8))


def chi_square_pvalue(reference: pd.Series, current: pd.Series) -> float:
    contingency = pd.concat(
        [
            reference.value_counts(normalize=True, dropna=False),
            current.value_counts(normalize=True, dropna=False),
        ],
        axis=1,
        keys=["reference", "current"],
    ).fillna(1e-6)

    _, p_value, _, _ = stats.chi2_contingency(contingency)
    return float(p_value)



In [ ]:
# Cálculo de métricas de drift por variable
records: List[Dict] = []

for col in numeric_cols:
    records.append(
        {
            "feature": col,
            "tipo": "numérica",
            "ks_stat": kolmogorov_smirnov(reference_df[col], current_df[col]),
            "psi": population_stability_index(reference_df[col], current_df[col]),
            "js_distance": jensen_shannon_distance(reference_df[col], current_df[col]),
            "chi2_pvalue": np.nan,
        }
    )

for col in categorical_cols:
    records.append(
        {
            "feature": col,
            "tipo": "categórica",
            "ks_stat": np.nan,
            "psi": np.nan,
            "js_distance": np.nan,
            "chi2_pvalue": chi_square_pvalue(reference_df[col], current_df[col]),
        }
    )

drift_df = pd.DataFrame(records)

# Flags simples de alerta
KS_THRESHOLD = 0.1
PSI_THRESHOLD = 0.25
PVAL_THRESHOLD = 0.05

conditions = []
conditions.append((drift_df["ks_stat"] > KS_THRESHOLD) & drift_df["ks_stat"].notna())
conditions.append((drift_df["psi"] > PSI_THRESHOLD) & drift_df["psi"].notna())
conditions.append((drift_df["chi2_pvalue"] < PVAL_THRESHOLD) & drift_df["chi2_pvalue"].notna())

alert_series = np.any(np.vstack([cond.fillna(False) for cond in conditions]), axis=0)
drift_df["alerta"] = alert_series

drift_df.sort_values(by=["alerta", "ks_stat", "psi"], ascending=[False, False, False]).reset_index(drop=True)


In [ ]:
# Visualización comparativa para una variable crítica
feature_to_plot = "Curricular units 1st sem (grade)"

fig = px.histogram(
    pd.DataFrame({
        "valor": pd.concat([reference_df[feature_to_plot], current_df[feature_to_plot]]),
        "dataset": ["referencia"] * len(reference_df) + ["producción"] * len(current_df),
    }),
    x="valor",
    color="dataset",
    barmode="overlay",
    nbins=30,
    title=f"Distribución comparativa - {feature_to_plot}",
)
fig.update_traces(opacity=0.65)
fig.show()



In [ ]:
# Simulación de tendencia temporal del accuracy (ejemplo)
np.random.seed(config["model_config"]["random_state"])
window_size = max(20, int(len(current_df) * 0.1))
rolling_accuracy = []
indices = []

for start in range(0, len(current_df), window_size):
    end = min(start + window_size, len(current_df))
    if end - start < 5:
        break
    chunk_y = current_y.iloc[start:end]
    chunk_pred = y_pred_current[start:end]
    rolling_accuracy.append(accuracy_score(chunk_y, chunk_pred))
    indices.append(start)

temporal_df = pd.DataFrame({"batch_inicio": indices, "accuracy": rolling_accuracy})
fig = px.line(temporal_df, x="batch_inicio", y="accuracy", title="Tendencia simulada del accuracy en producción")
fig.add_hline(y=perf_reference, line_dash="dash", annotation_text="Meta (accuracy referencia)")
fig.show()



### Integración con Streamlit
El siguiente fragmento puede incorporarse en la aplicación de monitoreo para visualizar las métricas principales:

```python
import streamlit as st
st.title("Dashboard de Data Drift")

st.metric("Accuracy referencia", f"{perf_reference:.3f}")
st.metric("Accuracy producción", f"{perf_current:.3f}")

st.dataframe(drift_df[["feature", "tipo", "ks_stat", "psi", "chi2_pvalue", "alerta"]])
st.plotly_chart(fig)
```

> Recomendación: programar la ejecución de este notebook (o script equivalente) en un job de orquestación (Airflow, Jenkins) y enviar alertas automáticas cuando `alerta == True` para cualquier variable crítica.


In [ ]:
# Persistencia de resultados de monitoreo
drift_report_path = outputs_dir / "drift_report.parquet"
drift_df.to_parquet(drift_report_path, index=False)

drift_report_path


## Próximos pasos
- Programar la ejecución periódica (diaria/semanal) del monitoreo y almacenar el histórico de métricas.
- Definir umbrales específicos por variable junto con el equipo de negocio.
- Integrar alertas vía correo o Slack cuando se detecte un `alerta == True`.
- Complementar con monitoreo de desempeño (precision, recall) a medida que se obtengan etiquetas reales en producción.
- Incorporar dashboards en Streamlit o BI corporativo siguiendo el fragmento de código sugerido.


o